**Accedemos a los datos ya tratados**

In [0]:
# Celda 1: Importación de librerías y lectura de Vuelos Silver
import pyspark.sql.functions as F

# Leemos directamente desde el catálogo Delta que creamos en el script anterior
df_vuelos_silver = spark.read.table("workspace.default.silver_vuelos_cancelaciones")

**Vemos cuantos aeropuertos tenemos para luego ver su condición del clima a lo largo de los periodos a analizar**

In [0]:
# Celda 2: Aislamiento de Aeropuertos de Origen Únicos
df_aeropuertos_unicos = df_vuelos_silver \
    .select("OriginAirportID", "Origin") \
    .distinct()

# Monitoreo de control
print(f"Cantidad total de aeropuertos comerciales identificados: {df_aeropuertos_unicos.count()}")

**Verificamos si al dejar destino por fuera del análisis anterior podría ser perjudicial ya que podría dejar aeropuertos por fuera. Sin embargo, se observa que no fue así, ya que en parte no tiene sentido que un aeropuerto solo despegue vuelos y no reciba...**

In [0]:
# Celda de Validación: Comparativa de Aeropuertos Origen vs Destino

# 1. Sacamos la lista única de aeropuertos de Origen
df_orígenes = df_vuelos_silver.select("OriginAirportID", "Origin").distinct()

# 2. Sacamos la lista única de aeropuertos de Destino
# Renombramos las columnas temporalmente para poder compararlas de forma idéntica
df_destinos = df_vuelos_silver.select(
    F.col("DestAirportID").alias("OriginAirportID"), 
    F.col("Dest").alias("Origin")
).distinct()

# 3. Operación de Conjuntos: Busquemos si hay aeropuertos que están en Destino pero NO en Origen
df_solo_en_destino = df_destinos.subtract(df_orígenes)

# 4. Imprimimos los resultados de la auditoría
print(f"Cantidad de aeropuertos únicos detectados en ORIGEN: {df_orígenes.count()}")
print(f"Cantidad de aeropuertos únicos detectados en DESTINO: {df_destinos.count()}")
print(f"Aeropuertos fantasmas (Están en destino pero nunca originaron un vuelo): {df_solo_en_destino.count()}")

# Si el conteo es mayor a 0, mostramos cuáles son esos aeropuertos
if df_solo_en_destino.count() > 0:
    print("\n--- Lista de aeropuertos que solo reciben vuelos ---")
    df_solo_en_destino.show()
else:
    print("\n¡Confirmado! Todos los aeropuertos que reciben vuelos también originan salidas.")

**Extraemos los aeropuertos del mundo para ver la ubicación de los que coinciden con los que se posee en el dataset**

In [0]:
# Celda 3: Descarga e Ingesta del Catálogo Geográfico Global Real
import urllib.request
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# 1. Definimos el esquema oficial de la base de datos de aeropuertos de OpenFlights
airports_schema = StructType([
    StructField("Airport_ID", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("IATA", StringType(), True), # Este es el código de 3 letras (JFK, LAX, etc.)
    StructField("ICAO", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("Altitude", StringType(), True),
    StructField("Timezone", StringType(), True),
    StructField("DST", StringType(), True),
    StructField("Tz_Database", StringType(), True),
    StructField("Type", StringType(), True),
    StructField("Source", StringType(), True)
])

# 2. Descargamos el dataset global real directamente desde el repositorio oficial de OpenFlights
url_catálogo = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat"
path_destino = "/tmp/airports_global.csv"

print("Descargando catálogo geográfico global desde OpenFlights...")
urllib.request.urlretrieve(url_catálogo, path_destino)

# 3. Lo leemos en Spark sin inferir esquema para mantener el rendimiento serverless
df_global_airports = spark.read \
    .format("csv") \
    .schema(airports_schema) \
    .load(f"file://{path_destino}")

# 4. Filtramos para quedarnos únicamente con los aeropuertos comerciales de EE.UU.
df_us_airports = df_global_airports.filter(F.col("Country") == "United States") \
                                   .select(F.col("IATA").alias("IATA_CODE"), "Latitude", "Longitude")


**Celda 4: Descarga e ingesta del catálogo geográfico en Volumen Bronze**

Esta celda descarga el catálogo global de aeropuertos desde OpenFlights directamente al volumen Bronze del Lakehouse, lo lee en Spark con el esquema oficial y filtra solo los aeropuertos comerciales de EE.UU. para obtener sus coordenadas (latitud/longitud). Así, se garantiza una copia persistente y controlada del catálogo geográfico en el entorno de datos.

In [0]:
# Celda 4: Descarga e Ingesta del Catálogo Geográfico en Volumen Bronze
import urllib.request
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# 1. Definimos el esquema oficial de OpenFlights
airports_schema = StructType([
    StructField("Airport_ID", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("IATA", StringType(), True), 
    StructField("ICAO", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("Altitude", StringType(), True),
    StructField("Timezone", StringType(), True),
    StructField("DST", StringType(), True),
    StructField("Tz_Database", StringType(), True),
    StructField("Type", StringType(), True),
    StructField("Source", StringType(), True)
])

# 2. Definimos la ruta de destino dentro de tu VOLUMEN BRONZE oficial
# Usamos la ruta local de la API POSIX de Databricks para guardar el archivo físicamente
path_descarga_local = "/Volumes/workspace/default/bronze_vuelos/airports_global.csv"

print("Descargando catálogo geográfico real directo a tu Volumen Bronze...")
url_catálogo = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat"
urllib.request.urlretrieve(url_catálogo, path_descarga_local)

# 3. Lo leemos en Spark de forma nativa desde el volumen (Sin prefijos raros de file://)
df_global_airports = spark.read \
    .format("csv") \
    .schema(airports_schema) \
    .load(path_descarga_local)

# 4. Filtramos para dejarnos los aeropuertos comerciales de EE.UU.
df_us_airports = df_global_airports.filter(F.col("Country") == "United States") \
                                   .select(F.col("IATA").alias("IATA_CODE"), "Latitude", "Longitude")

print("¡Catálogo de EE.UU. extraído con éxito en el Data Lakehouse!")

**Cruzamos los aeropuertos de nuestro dataset con el catálogo geográfico para obtener las coordenadas específicas (latitud/longitud) que nos permitirán consultar el clima.**

In [0]:
# ------------------------------------------------
# CELDA 5: REGENERACIÓN, JOIN Y CONTROL DE CALIDAD
# ------------------------------------------------

# 1. Regeneramos el DataFrame único directo del Lakehouse Silver para evitar el NameError
df_vuelos_silver = spark.read.table("workspace.default.silver_vuelos_cancelaciones")
df_aeropuertos_unicos = df_vuelos_silver.select("OriginAirportID", "Origin").distinct()

# 2. Ejecutamos el JOIN con el catálogo descargado en la Celda 4
df_aeropuertos_coordenadas = df_aeropuertos_unicos.join(
    df_us_airports,
    df_aeropuertos_unicos.Origin == df_us_airports.IATA_CODE,
    "inner"
).select(
    F.col("OriginAirportID"),
    F.col("Origin"),
    F.col("Latitude"),
    F.col("Longitude")
)

# 3. Métricas de control de integridad del cruce
total_origen = df_aeropuertos_unicos.count()
total_cruzados = df_aeropuertos_coordenadas.count()

print(f"Total de tus aeropuertos analíticos en el Lakehouse: {total_origen}")
print(f"Total de aeropuertos geolocalizados exitosamente: {total_cruzados}")

# 4. Aislamos los códigos huérfanos si los conteos no coinciden
if total_origen != total_cruzados:
    print(f"\n Alerta: Quedaron {total_origen - total_cruzados} aeropuertos sin coordenadas.")
    df_no_encontrados = df_aeropuertos_unicos.join(
        df_us_airports, 
        df_aeropuertos_unicos.Origin == df_us_airports.IATA_CODE, 
        "left_anti"
    )
    print("Códigos que hacen falta en el catálogo actual:")
    df_no_encontrados.show()
else:
    print("\n¡Éxito total! El 100% de tus aeropuertos se geolocalizaron de forma perfecta.")

display(df_aeropuertos_coordenadas.limit(10))

 **PERSISTENCIA GEOGRÁFICA: RECONSTRUCCIÓN DEL CATÁLOGO DE COORDENADAS (SILVER)
 Propósito:** Recupera los aeropuertos analíticos de los vuelos y les asigna sus
 coordenadas (latitud/longitud) usando el archivo maestro del volumen Bronze.
 Guarda el mapa resultante como una tabla Delta física para independizar la capa
 Gold de la memoria volátil, protegiendo el historial climático ya extraído.**


In [0]:
# ----------------------------------------------------------------------------
# PERSISTENCIA ADICIONAL SEGURO Y AUTÓNOMO SIN DEPENDER DE LA RAM DE ARRIBA
# ------------------------------------------------------------------------
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

target_geo_table = "workspace.default.silver_aeropuertos_coordenadas"
path_volumen_bronze = "/Volumes/workspace/default/bronze_vuelos/airports_global.csv"

print(" Paso 1: Re leyendo el catálogo de OpenFlights desde el Volumen Bronze...")

# 1. Volvemos a instanciar el esquema rígido para OpenFlights
airports_schema = StructType([
    StructField("Airport_ID", StringType(), True), StructField("Name", StringType(), True),
    StructField("City", StringType(), True), StructField("Country", StringType(), True),
    StructField("IATA", StringType(), True), StructField("ICAO", StringType(), True),
    StructField("Latitude", DoubleType(), True), StructField("Longitude", DoubleType(), True),
    StructField("Altitude", StringType(), True), StructField("Timezone", StringType(), True),
    StructField("DST", StringType(), True), StructField("Tz_Database", StringType(), True),
    StructField("Type", StringType(), True), StructField("Source", StringType(), True)
])

# 2. Leemos directo del volumen bronze
df_us_airports_local = spark.read \
    .format("csv") \
    .schema(airports_schema) \
    .load(path_volumen_bronze) \
    .filter(F.col("Country") == "United States")

print(" Paso 2: Recuperando aeropuertos analíticos de tus vuelos...")

# 3. Traemos los aeropuertos únicos directo de tu tabla Silver física de vuelos
df_vuelos_silver = spark.read.table("workspace.default.silver_vuelos_cancelaciones")
df_aeropuertos_unicos = df_vuelos_silver.select("OriginAirportID", "Origin").distinct()

print(" Paso 3: Ejecutando JOIN geoespacial...")

# 4. Hacemos el JOIN utilizando las variables locales que acabamos de levantar de disco
df_aeropuertos_coordenadas = df_aeropuertos_unicos.join(
    df_us_airports_local,
    df_aeropuertos_unicos.Origin == df_us_airports_local.IATA,
    "inner"
).select(
    F.col("OriginAirportID"),
    F.col("Origin"),
    F.col("Latitude"),
    F.col("Longitude")
)

print(f" Paso 4: Persistiendo catálogo geográfico de respaldo en {target_geo_table}...")

# 5. Escritura Delta final con Overwrite
df_aeropuertos_coordenadas.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(target_geo_table)

print("\n ¡Éxito total! Catálogo espacial Delta consolidado.")

**Sincroniza y estructura el historial climático diario para cada fecha del calendario maestro en todos los aeropuertos analíticos.**

> *Esta celda 6 realiza la extracción masiva del clima histórico para cada aeropuerto y cada día del periodo, consultando la API de Open-Meteo para cada combinación. Construye un DataFrame limpio con las variables meteorológicas clave (temperatura, precipitación, nieve, viento) y lo deja listo para persistencia o análisis posterior.*

In [0]:
#-----------------------------------------------------------------
# CELDA 6 FINAL: EXTRACCIÓN CON ASIGNACIÓN ESTÁTICA EXPLÍCITA
# ----------------------------------------------------------------
import requests
import time
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
import pyspark.sql.functions as F

aeropuertos_lista = df_aeropuertos_coordenadas.select("OriginAirportID", "Origin", "Latitude", "Longitude").collect()

start_date_str = "2025-01-01"
end_date_str = "2026-02-28" 

start_dt = datetime.strptime(start_date_str, "%Y-%m-%d")
end_dt = datetime.strptime(end_date_str, "%Y-%m-%d")
total_dias = (end_dt - start_dt).days + 1
lista_fechas_maestra = [(start_dt + timedelta(days=d)).strftime("%Y-%m-%d") for d in range(total_dias)]

print(f"Iniciando extracción estricta para {len(aeropuertos_lista)} aeropuertos...")

clima_rows = []
procesados = 0

for aero in aeropuertos_lista:
    id_aero = int(aero["OriginAirportID"])
    codigo = str(aero["Origin"])
    lat = aero["Latitude"]
    lon = aero["Longitude"]
    
    url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={start_date_str}&end_date={end_date_str}&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,wind_speed_10m_max&timezone=America%2FNew_York"
    
    clima_dict_aero = {}
    
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            data = response.json()
            daily_data = data.get("daily", {})
            
            fechas_api = daily_data.get("time", [])
            t_max = daily_data.get("temperature_2m_max", [])
            t_min = daily_data.get("temperature_2m_min", [])
            lluvia = daily_data.get("precipitation_sum", [])
            nieve = daily_data.get("snowfall_sum", [])
            viento = daily_data.get("wind_speed_10m_max", [])
            
            for i in range(len(fechas_api)):
                f_key = str(fechas_api[i]).strip()
                clima_dict_aero[f_key] = {
                    "t_max": float(t_max[i]) if t_max[i] is not None else None,
                    "t_min": float(t_min[i]) if t_min[i] is not None else None,
                    "lluvia": float(lluvia[i]) if lluvia[i] is not None else None,
                    "nieve": float(nieve[i]) if nieve[i] is not None else None,
                    "viento": float(viento[i]) if viento[i] is not None else None
                }
    except Exception as e:
        pass # Los aeropuertos con error se llenarán con nulos en el calendario maestro
    
    # CONSTRUCCIÓN DE FILA EXPLÍCITA SIN ASTERISCOS
    for fecha in lista_fechas_maestra:
        if fecha in clima_dict_aero:
            info = clima_dict_aero[fecha]
            clima_rows.append((
                id_aero,
                codigo,
                fecha,
                info["t_max"],
                info["t_min"],
                info["lluvia"],
                info["nieve"],
                info["viento"]
            ))
        else:
            clima_rows.append((id_aero, codigo, fecha, None, None, None, None, None))
            
    time.sleep(0.15)
    procesados += 1
    if procesados % 100 == 0 or procesados == len(aeropuertos_lista):
        print(f"Sincronizados {procesados}/{len(aeropuertos_lista)} aeropuertos.")

# Esquema de Spark estricto y limpio
clima_schema = StructType([
    StructField("AirportID", IntegerType(), False),
    StructField("AirportCode", StringType(), False),
    StructField("Date_String", StringType(), False),
    StructField("MaxTemperature", DoubleType(), True),
    StructField("MinTemperature", DoubleType(), True),
    StructField("Precipitation", DoubleType(), True),
    StructField("Snowfall", DoubleType(), True),
    StructField("MaxWindSpeed", DoubleType(), True)
])

# Construcción y parseo en Spark
df_clima_raw = spark.createDataFrame(clima_rows, schema=clima_schema)
df_clima_silver = df_clima_raw.withColumn("WeatherDate", F.to_date(F.col("Date_String"), "yyyy-MM-dd")) \
                              .withColumn("Year", F.year(F.col("WeatherDate"))) \
                              .withColumn("Month", F.month(F.col("WeatherDate"))) \
                              .drop("Date_String")

# Verificación de Control de Calidad
nulos_temperatura = df_clima_silver.filter(F.col("MaxTemperature").isNull()).count()
print(f"\n[CONTROL DE CALIDAD] Filas totales: {df_clima_silver.count()} | Filas con Temperatura Nula: {nulos_temperatura}")

display(df_clima_silver.filter(F.col("MaxTemperature").isNotNull()).limit(5))


**Celda 7: Captura directa y escritura en Silver**

Esta celda toma el DataFrame `df_clima_silver` con el historial climático diario de todos los aeropuertos y lo persiste como una tabla Delta física particionada por año y mes (`silver_clima_historico`). Así, asegura la disponibilidad y durabilidad del historial meteorológico para análisis posteriores, independizándolo de la memoria RAM y facilitando consultas eficientes.

In [0]:
# --------------------------------------------------------------------------------
# CELDA 7 BLINDADA: CAPTURA DIRECTA Y ESCRITURA EN CAPA SILVER
# ------------------------------------------------------------------------
target_clima_table = "workspace.default.silver_clima_historico"

# Forzamos la validación de la variable global de la sesión activa
try:
    if 'df_clima_silver' in locals() or 'df_clima_silver' in globals():
        print(f"Persistiendo registros en {target_clima_table}...")
        
        df_clima_silver.write \
            .format("delta") \
            .mode("overwrite") \
            .partitionBy("Year", "Month") \
            .saveAsTable(target_clima_table)
            
        print("¡Éxito absoluto! La tabla 'silver_clima_historico' ha sido consolidada.")
    else:
        print(" La variable local se limpió. Ejecuta un segundo la Celda 6 de arriba para refrescar la RAM.")
except Exception as e:
    print(f" Error durante la persistencia Delta: {str(e)}")


**Celda 8: Auditoría de Calidad y Persistencia en Silver**

Esta celda realiza una auditoría exhaustiva sobre la tabla Delta `silver_clima_historico` para validar la integridad y calidad del historial meteorológico persistido. Evalúa el volumen de registros, la cobertura de aeropuertos, el rango temporal, la presencia de valores nulos en variables clave y la correcta partición física por año y mes. Así, garantiza que la capa Silver esté lista para análisis avanzados y procesos posteriores en la arquitectura del Lakehouse.

In [0]:
# -----------------------------------------------------------------------
# CELDA 8: AUDITORÍA DE CALIDAD Y PERSISTENCIA (CAPA SILVER DELTA)
# ------------------------------------------------------------------------
import pyspark.sql.functions as F

target_clima_table = "workspace.default.silver_clima_historico"

print(f"--- INICIANDO AUDITORÍA DE CALIDAD SOBRE METADATOS DELTA ---")

# 1. Cargamos el DataFrame directo desde el Unity Catalog
df_auditoria = spark.read.table(target_clima_table)

# 2. Conteo Físico Real en Disco
total_filas_disco = df_auditoria.count()
print(f" Volumen de registros persistidos en Delta: {total_filas_disco}")

# 3. Verificación de Integridad de Aeropuertos
aeropuertos_distintos = df_auditoria.select("AirportCode").distinct().count()
print(f" Total de identificadores aéreos guardados: {aeropuertos_distintos} de 347 esperados.")

# 4. Auditoría de Distribución Temporal (Fechas Extreman)
fechas_limites = df_auditoria.select(
    F.min("WeatherDate").alias("Fecha_Inicio"), 
    F.max("WeatherDate").alias("Fecha_Fin")
).collect()

print(f" Ventana temporal asegurada en disco: Desde {fechas_limites[0]['Fecha_Inicio']} hasta {fechas_limites[0]['Fecha_Fin']}")

# 5. Mapeo de Nulos Residuales por columna (Para la estrategia de la Capa Gold)
print(f"\n--- MAPEO FÍSICO DE HUECOS METEOROLÓGICOS (NULOS) ---")
df_auditoria.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in ["MaxTemperature", "MinTemperature", "Precipitation", "Snowfall", "MaxWindSpeed"]
]).show()

# 6. Distribución de datos por Año/Mes (Validación de partición física)
print("--- DISTRIBUCIÓN DE FILAS POR PARTICIÓN (AÑO/MES) ---")
df_auditoria.groupBy("Year", "Month") \
            .count() \
            .orderBy("Year", "Month") \
            .show(15)